# Baseline 3: prompted intent parser (GPU step)

Qwen2.5-7B-Instruct (4-bit) reads each task's request + dataset schema and writes an intent contract (vocabulary v1.0).
Only **parsing** runs here. Validation, contract metrics and the verifier run on the laptop.

**Settings:** Accelerator **GPU T4 x1**, Internet **on**. Private repo: Kaggle secret `GITHUB_TOKEN`.
Then *Save Version → Save & Run All* (about 30–60 min for 120 tasks). Download `parse_<model>.zip` from the Output tab.

Prompt `p1-v1`: few-shot examples are 5 **dev** tasks (one per transformation type); greedy decoding, no constrained decoding (README §12).

In [ ]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/<user>/<repo>.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
LOAD_IN_4BIT = True       # 7B in fp16 doesn't fit a 16 GB T4
BATCH = 4
LIMIT = None              # e.g. 4 for a quick dry run
TAG = MODEL_ID.split("/")[-1].lower().replace(".", "")
OUT = f"/kaggle/working/parse_{TAG}"
print(TAG, OUT)

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s (%cd)"], capture_output=True, text=True).stdout)
NEEDED = ["src/igt/agents/intent_parser.py", "src/igt/contracts/schema.py", "eval/parse_contracts.py",
          "eval/generate.py", "eval/prompts/schema_b1-v1.json", "eval/tasks"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"Not in the GitHub repo (push them first): {missing}"
print("all needed files present")

In [ ]:
# ---------------- Libraries: bitsandbytes for 4-bit (pydantic 2 is needed for validation) ----------------
if LOAD_IN_4BIT:
    subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
import sys, torch, pydantic
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator")
print("pydantic", pydantic.VERSION)
assert pydantic.VERSION.startswith("2"), "pydantic 2 required"
sys.path.insert(0, "src")
from igt.agents.intent_parser import PROMPT_HASH, PARSER_PROMPT_VERSION
print("parser prompt:", PARSER_PROMPT_VERSION, PROMPT_HASH)

In [ ]:
# ---------------- Parse every task (resumable) ----------------
cmd = ["python", "eval/parse_contracts.py", "--model", MODEL_ID, "--out", OUT, "--batch", str(BATCH)]
if LOAD_IN_4BIT:
    cmd.append("--load-in-4bit")
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-4000:])
print(p.stderr[-3000:])
assert p.returncode == 0

In [ ]:
# ---------------- Quick look and zip ----------------
import json, collections
R = [json.loads(l) for l in open(f"{OUT}/parses.jsonl")]
print("parses:", len(R), "| status:", collections.Counter(r["parse_status"] for r in R), "| truncated:", sum(r["truncated"] for r in R))
print(R[0]["task_id"], "->"); print(R[0]["raw_output"][:1500])
open(f"{OUT}/git_commit.txt", "w").write(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("saved:", shutil.make_archive(OUT, "zip", OUT))
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)